# Validation Set Viewer

Browse held-out validation samples and compare **Prior** (OpenUtau), **SoulX Target** (ground truth), and **Generated** outputs from **multiple checkpoints** side-by-side.

In [ ]:
# ── Configuration ──

CHECKPOINTS = {
    "4-19 PL-BERT FT": "../AdversarialFinetune/checkpoints/4-19-pl-bert-ft/checkpoint_85000.pt",
    "4-23 Embed FT": "../AdversarialFinetune/checkpoints/4-23-embed-ft/checkpoint_80000.pt",
}

MANIFEST_PATH = "../Data/Rachie/manifest.csv"
DATA_DIR = "../Data/Rachie"

N_SAMPLES = 8
DEVICE = "auto"  # "auto", "cuda", or "cpu"

# Validation split (must match training config for deterministic reproduction)
MAX_DTW_COST = 100.0
VAL_FRACTION = 0.05
SEED = 42

# ODE inference
NUM_ODE_STEPS = 4
ODE_METHOD = "midpoint"
CFG_SCALE = 2.0
CHUNK_SIZE = 256
OVERLAP = 16

# Mel spectrogram constants (SoulX-Singer z-score normalization)
MEL_MEAN = -4.92
MEL_VAR = 8.14
SR = 24000
HOP = 480

In [ ]:
import sys, os, math
from pathlib import Path

import numpy as np
import torch
import matplotlib.pyplot as plt
from IPython.display import Audio, display, HTML

# ── Path setup (mirrors VocaloFlow/inference/pipeline.py:28-45) ──
REPO_ROOT = str(Path("..").resolve())
VOCALOFLOW_DIR = os.path.join(REPO_ROOT, "VocaloFlow")
SOULX_DIR = os.path.join(REPO_ROOT, "SoulX-Singer")

for p in [REPO_ROOT, VOCALOFLOW_DIR, SOULX_DIR]:
    if p not in sys.path:
        sys.path.insert(0, p)

from utils.data_helpers import load_manifest, filter_manifest, split_by_song
from utils.resample import resolve_phoneme_indirection
from inference.pipeline import load_model, infer_chunked, mel_to_wav

# ── Resolve device ──
if DEVICE == "auto":
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
else:
    device = torch.device(DEVICE)
print(f"Using device: {device}")


# ── Helpers ──

def denormalize_mel(mel):
    return mel * math.sqrt(MEL_VAR) + MEL_MEAN


def load_chunk_inputs(row, use_plbert):
    """Load all model inputs for a single validation chunk (full length, no cropping)."""
    prior_mel = np.load(row["prior_mel_path"]).astype(np.float32)
    target_mel = np.load(row["target_mel_path"]).astype(np.float32)
    f0 = np.load(row["f0_path"]).astype(np.float32)
    voicing = np.load(row["voicing_path"]).astype(np.float32)

    chunk_dir = os.path.dirname(row["prior_mel_path"])
    phoneme_mask = np.load(row["phoneme_mask_path"]).astype(np.int64)
    phoneme_ids_raw = np.load(os.path.join(chunk_dir, "phoneme_ids.npy")).astype(np.int64)
    resolved = resolve_phoneme_indirection(phoneme_ids_raw, phoneme_mask)

    T = target_mel.shape[0]

    def match_1d(x, n):
        return x[:n] if len(x) >= n else np.pad(x, (0, n - len(x)))

    def match_2d(x, n):
        return x[:n] if x.shape[0] >= n else np.pad(x, ((0, n - x.shape[0]), (0, 0)))

    prior_mel = match_2d(prior_mel, T)
    f0 = match_1d(f0, T)
    voicing = match_1d(voicing, T)
    resolved = match_1d(resolved, T)

    result = {
        "prior_mel": prior_mel,
        "target_mel": target_mel,
        "f0": f0,
        "voicing": voicing,
        "phoneme_ids": resolved,
        "chunk_dir": chunk_dir,
    }

    if use_plbert:
        plbert_path = os.path.join(chunk_dir, "plbert_features.npy")
        if os.path.exists(plbert_path):
            plbert_feats = np.load(plbert_path).astype(np.float32)
            mask_clipped = np.clip(phoneme_mask, 0, len(plbert_feats) - 1)
            plbert_frame = plbert_feats[mask_clipped]
            plbert_frame = match_2d(plbert_frame, T)
            result["plbert_features"] = plbert_frame
        else:
            print(f"  WARNING: plbert_features.npy not found in {chunk_dir}")
            result["plbert_features"] = None

    return result

In [ ]:
# ── Load manifest and reproduce validation split ──

manifest = load_manifest(MANIFEST_PATH, DATA_DIR)
filtered = filter_manifest(manifest, max_dtw_cost=MAX_DTW_COST)
train_df, val_df = split_by_song(filtered, val_fraction=VAL_FRACTION, seed=SEED)

n_display = min(N_SAMPLES, len(val_df))
print(f"\nValidation set: {len(val_df)} chunks from {val_df['dali_id'].nunique()} songs")
print(f"Displaying {n_display} samples")

In [ ]:
# ── Load models ──

models = {}
speaker_embeddings = {}

for label, ckpt_path in CHECKPOINTS.items():
    print(f"\n── Loading: {label} ({ckpt_path})")
    model = load_model(ckpt_path, device)
    models[label] = model

    use_plbert = model.config.use_plbert
    print(f"  PL-BERT: {use_plbert}")

    spk_emb = None
    if model.config.use_speaker_embedding:
        spk_path = os.path.join(REPO_ROOT, "SpeakerEmbedding", "embeddings", "Rachie", "speaker_embedding.pt")
        if os.path.exists(spk_path):
            spk_emb = torch.load(spk_path, weights_only=True).float().unsqueeze(0).to(device)
            print(f"  Speaker embedding: {spk_emb.shape}")
        else:
            print(f"  WARNING: speaker embedding not found at {spk_path}")
    else:
        print("  Speaker embedding: disabled")
    speaker_embeddings[label] = spk_emb

# Use the superset of plbert requirements across all models
any_use_plbert = any(m.config.use_plbert for m in models.values())
print(f"\nLoaded {len(models)} models. Any needs PL-BERT: {any_use_plbert}")

In [ ]:
# ── Display validation samples ──

import soundfile as sf

labels = list(models.keys())
n_models = len(labels)

for i in range(n_display):
    row = val_df.iloc[i]

    display(HTML(
        f"<h3>Sample {i + 1}/{n_display}: "
        f"<code>{row['dali_id']}</code> / <code>{row['chunk_name']}</code></h3>"
    ))
    print(f"DTW cost: {row['dtw_cost']:.2f}")

    try:
        inputs = load_chunk_inputs(row, any_use_plbert)
        prior_mel = inputs["prior_mel"]
        target_mel = inputs["target_mel"]

        # ── Run inference for each checkpoint ──
        output_mels = {}
        for label in labels:
            model = models[label]
            plbert = inputs.get("plbert_features") if model.config.use_plbert else None
            with torch.no_grad():
                output_mels[label] = infer_chunked(
                    model, prior_mel, inputs["f0"], inputs["voicing"],
                    inputs["phoneme_ids"],
                    chunk_size=CHUNK_SIZE, overlap=OVERLAP,
                    num_steps=NUM_ODE_STEPS, method=ODE_METHOD,
                    device=device, cfg_scale=CFG_SCALE,
                    plbert_features=plbert,
                    speaker_embedding=speaker_embeddings[label],
                )

        # ── Mel spectrograms: Prior, Target, then each checkpoint ──
        n_panels = 2 + n_models
        all_mels = [prior_mel, target_mel] + [output_mels[l] for l in labels]
        all_titles = ["Prior (OpenUtau)", "SoulX Target"] + [f"Generated: {l}" for l in labels]
        vmin = min(m.min() for m in all_mels)
        vmax = max(m.max() for m in all_mels)

        fig, axes = plt.subplots(1, n_panels, figsize=(8 * n_panels, 5))
        fig.suptitle(f"{row['dali_id']} / {row['chunk_name']}", fontsize=14)
        for ax, mel, title in zip(axes, all_mels, all_titles):
            t_end = mel.shape[0] * HOP / SR
            im = ax.imshow(
                mel.T, origin="lower", aspect="auto", cmap="magma",
                vmin=vmin, vmax=vmax,
                extent=[0, t_end, 0, 128],
            )
            ax.set_title(title)
            ax.set_xlabel("Time (s)")
            ax.set_ylabel("Mel bin")
        fig.colorbar(im, ax=list(axes), shrink=0.8, label="Normalized amplitude")
        plt.tight_layout()
        plt.show()

        # ── Difference plots: each checkpoint vs target ──
        fig, axes = plt.subplots(1, n_models, figsize=(8 * n_models, 4))
        if n_models == 1:
            axes = [axes]
        for ax, label in zip(axes, labels):
            diff = output_mels[label] - target_mel
            vlim = np.abs(diff).max()
            t_end = diff.shape[0] * HOP / SR
            im = ax.imshow(
                diff.T, origin="lower", aspect="auto",
                cmap="coolwarm", vmin=-vlim, vmax=vlim,
                extent=[0, t_end, 0, 128],
            )
            ax.set_title(f"Δ Target: {label}")
            ax.set_xlabel("Time (s)")
            ax.set_ylabel("Mel bin")
            fig.colorbar(im, ax=ax, label="Δ amplitude")
            print(f"  [{label}] mean |Δ| = {np.abs(diff).mean():.4f},  max |Δ| = {vlim:.4f}")
        plt.tight_layout()
        plt.show()

        # ── Audio playback ──
        prior_wav_path = os.path.join(inputs["chunk_dir"], "prior.wav")
        target_wav_path = os.path.join(inputs["chunk_dir"], "target.wav")

        print("Prior:")
        display(Audio(prior_wav_path))
        print("SoulX Target:")
        display(Audio(target_wav_path))

        for label in labels:
            print(f"Generated ({label}):")
            gen_audio = mel_to_wav(output_mels[label])
            display(Audio(gen_audio, rate=SR))

    except Exception as e:
        print(f"ERROR processing sample {i}: {e}")

    display(HTML("<hr>"))

## Notes

- Add entries to `CHECKPOINTS` dict (`"label": "path"`) to compare any number of models
- Mel spectrograms show Prior, Target, then one panel per checkpoint
- Difference plots and mean |Δ| are shown per checkpoint for easy comparison
- Audio playback is listed sequentially: Prior, Target, then each checkpoint
- The validation split is deterministic (seed=42, song-level) and matches training